# Custom Vectorized Autograd and Deep Learning Engine From Scratch

This notebook implements a dynamic, fully vectorized autograd engine and multi-layer perceptron (MLP) built from the ground up using only `numpy`. It mimics the core architectural mechanics of frameworks like PyTorch (`torch.Tensor` and `torch.autograd`), supporting custom model topology, dynamic shapes, backward execution graphs, and runtime hyperparameter tuning.

In [6]:
#Imports and Tensor Engine

import numpy as np

class Tensor:
    def __init__(self, data, _children=(), _op=''):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Tensor(shape={self.data.shape}, op={self._op})"

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')

        def _backward():
            self_grad = out.grad.copy()
            other_grad = out.grad.copy()
            while self_grad.ndim > self.data.ndim:
                self_grad = self_grad.sum(axis=0)
            for axis, size in enumerate(self.data.shape):
                if size == 1:
                    self_grad = self_grad.sum(axis=axis, keepdims=True)
            while other_grad.ndim > other.data.ndim:
                other_grad = other_grad.sum(axis=0)
            for axis, size in enumerate(other.data.shape):
                if size == 1:
                    other_grad = other_grad.sum(axis=axis, keepdims=True)
            self.grad += self_grad
            other.grad += other_grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), '*')

        def _backward():
            self_grad = (other.data * out.grad).copy()
            other_grad = (self.data * out.grad).copy()
            while self_grad.ndim > self.data.ndim: self_grad = self_grad.sum(axis=0)
            for axis, size in enumerate(self.data.shape):
                if size == 1: self_grad = self_grad.sum(axis=axis, keepdims=True)
            while other_grad.ndim > other.data.ndim: other_grad = other_grad.sum(axis=0)
            for axis, size in enumerate(other.data.shape):
                if size == 1: other_grad = other_grad.sum(axis=axis, keepdims=True)
            self.grad += self_grad
            other.grad += other_grad
        out._backward = _backward
        return out

    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other), '@')
        def _backward():
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad
        out._backward = _backward
        return out

    def sum(self):
        out = Tensor(np.sum(self.data), (self,), 'sum')
        def _backward():
            self.grad += np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = np.tanh(self.data)
        out = Tensor(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1.0 / (1.0 + np.exp(-np.clip(self.data, -500, 500)))
        out = Tensor(s, (self,), 'sigmoid')
        def _backward():
            self.grad += (s * (1.0 - s)) * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "Power only supports int/float"
        out = Tensor(self.data**other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * (self.data**(other - 1))) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = np.ones_like(self.data)
        for node in reversed(topo):
            node._backward()

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)

## Neural Network Modular Layers

The blocks below implement structural layers by extending a base `Module` parent class. It leverages our custom matrix multiplication (`@`) operator within the linear transformations, supporting dynamic input/output dimensions and parameter tracking.

In [7]:
#Neural Network Structs

class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = np.zeros_like(p.data)
    def parameters(self):
        return []

class Linear(Module):
    def __init__(self, nin, nout, act_type='tanh'):
        self.w = Tensor(np.random.randn(nin, nout) * 0.1)
        self.b = Tensor(np.zeros(nout))
        self.act_type = act_type

    def __call__(self, x):
        act = (x @ self.w) + self.b
        if self.act_type == 'tanh': return act.tanh()
        elif self.act_type == 'relu': return act.relu()
        elif self.act_type == 'sigmoid': return act.sigmoid()
        return act

    def parameters(self):
        return [self.w, self.b]

class MLP(Module):
    def __init__(self, nin, nouts, act_type='tanh'):
        sz = [nin] + nouts
        self.layers = []
        for i in range(len(nouts)):
            layer_act = act_type if i != len(nouts)-1 else 'linear'
            self.layers.append(Linear(sz[i], sz[i+1], act_type=layer_act))

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


## Optimization Objectives (Loss Functions)

We decouple our loss configurations into standard optimization metrics. Each loss function processes predictions against targets and propagates gradients back across the dynamic operational path via the underlying autograd engine.

In [8]:
#Loss Functions

def mse_loss(pred, target):
    return ((pred - target)**2).sum()

def mae_loss(pred, target):
    diff = pred - target
    out = Tensor(np.sum(np.abs(diff.data)), (pred, target), 'mae')
    def _backward():
        pred.grad += np.sign(diff.data) * out.grad
        target.grad += -np.sign(diff.data) * out.grad
    out._backward = _backward
    return out

## Interactive Pipeline Environment

Execute the cell below to configure your runtime parameters, input your data matrices manually, choose your activation functions, and track model weights and optimization behaviors in real-time.

In [9]:
print("=" * 60)
print("FULLY DYNAMIC ENGINE CONFIGURATOR")
print("=" * 60)

try:
    num_samples = int(input("Enter number of rows (samples): "))
    num_features = int(input("Enter number of columns (features): "))
    num_outputs = int(input("Enter number of target output dimensions: "))
    epochs = int(input("Enter number of epochs to train: "))
    
    num_hidden_layers = int(input("Enter number of hidden layers: "))
    hidden_architecture = []
    for h in range(num_hidden_layers):
        nodes = int(input(f"  Enter number of neurons for hidden layer {h+1}: "))
        hidden_architecture.append(nodes)
    hidden_architecture.append(num_outputs)
    
    chosen_act = input("Select activation function (tanh, relu, sigmoid): ").strip().lower()
    if chosen_act not in ['tanh', 'relu', 'sigmoid']:
        raise ValueError("Invalid activation function.")
        
    chosen_loss = input("Select loss function (mse, mae): ").strip().lower()
    if chosen_loss not in ['mse', 'mae']:
        raise ValueError("Invalid loss function.")

    print(f"\n[Matrix Expectations -> X: ({num_samples}, {num_features}) | Y: ({num_samples}, {num_outputs})]")
    
    raw_x = []
    for i in range(num_samples):
        row = input(f"  Enter row {i+1} inputs ({num_features} values separated by space): ").split()
        if len(row) != num_features:
            raise ValueError(f"Expected {num_features} items, got {len(row)}")
        raw_x.append([float(val) for val in row])
        
    raw_y = []
    for i in range(num_samples):
        target = input(f"  Enter row {i+1} target ({num_outputs} values separated by space): ").split()
        if len(target) != num_outputs:
            raise ValueError(f"Expected {num_outputs} items, got {len(target)}")
        raw_y.append([float(val) for val in target])
        
    X = Tensor(raw_x)
    Y = Tensor(raw_y)
    
    model = MLP(num_features, hidden_architecture, act_type=chosen_act)
    loss_fn = mse_loss if chosen_loss == 'mse' else mae_loss
    
    print("\nBeginning Optimization Run...")
    
    for k in range(1, epochs + 1):
        ypred = model(X)
        loss = loss_fn(ypred, Y)
        model.zero_grad()
        loss.backward()
        
        lr = 0.01
        for p in model.parameters():
            p.data -= lr * p.grad
            
        if k == 1 or k % 5 == 0 or k == epochs:
            print("\n" + "-" * 60)
            print(f"Epoch {k:<4} | Graph Loss: {loss.data:.4f}")
            print("-" * 60)
            for idx, layer in enumerate(model.layers):
                layer_name = f"Hidden Layer {idx+1}" if idx < len(model.layers)-1 else "Output Layer"
                print(f"[{layer_name} Weights]:")
                print(layer.w.data.round(5))
                
    print("\n" + "=" * 60)
    print("Training finalized.")
    print("=" * 60)
        
except ValueError as e:
    print(f"\n[Execution Error] Configuration verification failed: {e}")

FULLY DYNAMIC ENGINE CONFIGURATOR


Enter number of rows (samples):  3
Enter number of columns (features):  2
Enter number of target output dimensions:  1
Enter number of epochs to train:  50
Enter number of hidden layers:  3
  Enter number of neurons for hidden layer 1:  64
  Enter number of neurons for hidden layer 2:  32
  Enter number of neurons for hidden layer 3:  32
Select activation function (tanh, relu, sigmoid):  relu
Select loss function (mse, mae):  mse



[Matrix Expectations -> X: (3, 2) | Y: (3, 1)]


  Enter row 1 inputs (2 values separated by space):  2 3
  Enter row 2 inputs (2 values separated by space):  2 6
  Enter row 3 inputs (2 values separated by space):  9 8
  Enter row 1 target (1 values separated by space):  1
  Enter row 2 target (1 values separated by space):  3
  Enter row 3 target (1 values separated by space):  5



Beginning Optimization Run...

------------------------------------------------------------
Epoch 1    | Graph Loss: 32.8984
------------------------------------------------------------
[Hidden Layer 1 Weights]:
[[-0.02915  0.05886 -0.07186 -0.08637  0.091   -0.05394 -0.08715 -0.01875
   0.25783  0.12989  0.04415  0.02041 -0.15098 -0.17119  0.0386   0.12086
  -0.08132 -0.02178 -0.13574  0.0003   0.017    0.0022  -0.07549 -0.00988
  -0.12862 -0.16272 -0.14089 -0.16623 -0.17138 -0.04242  0.13892 -0.03867
   0.02151 -0.25716 -0.07056  0.09955 -0.02789 -0.0894   0.0083  -0.13789
   0.02466  0.10731 -0.07618 -0.13779 -0.05631  0.10668  0.00301 -0.01957
  -0.03902 -0.0701   0.1806   0.13189 -0.20706 -0.0682  -0.02825 -0.05885
  -0.22347 -0.10174 -0.11048  0.1481  -0.13163  0.09258  0.19903 -0.02078]
 [ 0.07936 -0.116   -0.06655 -0.05876 -0.15004  0.01257  0.08283  0.03304
  -0.06823  0.11324  0.03277 -0.15248 -0.021   -0.03555 -0.05794  0.20173
  -0.01042  0.04187  0.12091  0.2544  -0.1348 